In [ ]:
%pip install qiskit_aer
%pip install qiskit
%pip install qiskit_ibm_runtime
%pip install pylatexenc

import networkx as nx # Used for creating, manipulating, and studying the structure, dynamics, and functions of complex networks.
import matplotlib.pyplot as plt # Used for creating static, animated, and interactive visualizations in Python.
import numpy as np # Used for numerical operations, especially with arrays and matrices.
import itertools # Used for creating iterators for efficient looping.
from qiskit.circuit import QuantumCircuit, Parameter, ParameterVector # Core classes for building quantum circuits in Qiskit.
from qiskit.quantum_info import SparsePauliOp # Represents a sum of Pauli operators, used for Hamiltonians.

from qiskit import transpile # Function to optimize and map quantum circuits to a target backend.
from qiskit.circuit.library import PauliEvolutionGate # Used to construct a gate for evolving under a Pauli Hamiltonian.

from qiskit_aer import AerSimulator # A high-performance simulator backend for Qiskit.

from scipy.optimize import minimize # Used for finding the minimum of a scalar function of one or more variables.

from qiskit.result import marginal_counts # Used to extract counts for a subset of classical bits.
from qiskit.visualization import plot_histogram # Used for plotting histograms of measurement outcomes.
from qiskit.circuit.library import QAOAAnsatz # Pre-built QAOA ansatz circuit.
from qiskit_ibm_runtime import QiskitRuntimeService # Used to connect to and manage Qiskit Runtime services.

from qiskit_aer import AerSimulator # Re-import for clarity or specific usage in some contexts.
from qiskit_ibm_runtime import SamplerV2 as Sampler # Qiskit Runtime Sampler primitive for obtaining quasi-probability distributions.
from scipy.optimize import minimize # Re-import for clarity or specific usage in some contexts.
from qiskit_ibm_runtime import EstimatorV2 as Estimator # Qiskit Runtime Estimator primitive for calculating expectation values.

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager # Generates a preset pass manager for common transpilation scenarios.


**Create Total Cost/Penalty Hamiltonian**
\
\
Total Hamiltonian = Modularity Penalty + Supporting Penalty

In [ ]:
def create_total_cost_hamiltonian(cost_hamiltonian, penalty_hamiltonian, num_qubits):
      """Combines the Cost Hamiltonian and the Penalty Hamiltonian to form the total objective function.

      The total Hamiltonian represents the overall optimization problem, where the goal is to
      find a quantum state that minimizes its expectation value. This combined Hamiltonian
      includes terms that encourage desired community structures (Cost Hamiltonian) and
      terms that enforce problem constraints (Penalty Hamiltonian).

      Args:
            cost_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian representing the modularity objective.
            penalty_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian enforcing the one-hot encoding constraint.
            
      Returns:
            qiskit.quantum_info.SparsePauliOp: The combined total Hamiltonian.
      """
      # Combine the Cost Hamiltonian and Penalty Hamiltonian by summing them.
      # The objective is to minimize this combined Hamiltonian.
      total_hamiltonian = cost_hamiltonian + penalty_hamiltonian

      print(f"Total Hamiltonian constructed for {num_qubits} qubits:")
      print(total_hamiltonian)
      print(f"Number of terms in Total Hamiltonian: {len(total_hamiltonian)}")

      return total_hamiltonian

**Mixer Hamiltonian XY**

$H_M=\sum_{i=1}^{n}\sum_{r<s}\left(X_{i,r}X_{i,s}+Y_{i,r}Y_{i,s}\right)$


In [ ]:
def create_xy_mixer_hamiltonian(num_qubits, N, K):
    """Creates the mixer Hamiltonian (H_M) for the QAOA algorithm. Only for One-Hot

    The mixer Hamiltonian drives transitions between different computational basis states,
    helping the QAOA algorithm explore the solution space. It is typically a sum of
    single-qubit Pauli X operators or more complex operators like the XY mixer.
    This specific implementation uses the XY mixer, which promotes mixing between
    different community assignments for each node.

    H_M = sum_{i=1}^{n} sum_{r<s} (X_{i,r}X_{i,s} + Y_{i,r}Y_{i,s})

    Args:
        num_qubits (int): Total number of qubits in the system (N*K).
        N (int): The number of nodes in the original graph, used to structure the mixer terms.
        K (int): The number of communities, which determines how the mixer terms are structured
        
    Returns:
        qiskit.quantum_info.SparsePauliOp: The constructed XY Mixer Hamiltonian.
    """
    mixer_pauli_terms = []

    for i in range(N): # Iterate over each original node (0 to N-1)
        for r in range(K): # Iterate over community r
            for s in range(r + 1, K): # Iterate over community s, where s > r, to consider unique pairs
                qubit_idx_ir = i * K + r # Qubit for node i in community r
                qubit_idx_is = i * K + s # Qubit for node i in community s

                # X_ir X_is term: Applies Pauli X to qubits corresponding to node i in communities r and s
                pauli_string_x = ['I'] * num_qubits
                pauli_string_x[qubit_idx_ir] = 'X'
                pauli_string_x[qubit_idx_is] = 'X'
                mixer_pauli_terms.append(("".join(reversed(pauli_string_x)), 1.0))

                # Y_ir Y_is term: Applies Pauli Y to qubits corresponding to node i in communities r and s
                pauli_string_y = ['I'] * num_qubits
                pauli_string_y[qubit_idx_ir] = 'Y'
                pauli_string_y[qubit_idx_is] = 'Y'
                mixer_pauli_terms.append(("".join(reversed(pauli_string_y)), 1.0))

    mixer_hamiltonian_xy = SparsePauliOp.from_list(mixer_pauli_terms, num_qubits=num_qubits)

    print(f"\n\nXY Mixer Hamiltonian constructed for {num_qubits} qubits:")
    print(mixer_hamiltonian_xy)
    print(f"Number of terms in XY Mixer Hamiltonian: {len(mixer_pauli_terms)}")

    return mixer_hamiltonian_xy

In [ ]:
def build_w_state_initial_state(num_qubits, N, K):
    """Prepare the W-state on each node's K-qubit block.

    For node i, the K qubits are [i*K, i*K+1, ..., i*K+K-1].
    Each block is initialised to:
        |W_K> = (1/sqrt(K))(|10...0> + |01...0> + ... + |00...1>)

    Why this is the right initial state
    ------------------------------------
    The XY mixer e^{-ib H_M} preserves Hamming weight within each
    node's qubit block.  The W-state has Hamming weight exactly 1,
    which is the one-hot constraint.  Starting here guarantees the
    circuit never leaves the valid (one-hot) subspace, so the
    penalty Hamiltonian is not fighting the initial state.

    Circuit construction (per node block, K qubits)
    ------------------------------------------------
    1.  X on qubit 0  ->  |1, 0, ..., 0>
    2.  For i = 0 to K-2:
          a. CRy(2*arccos(1/sqrt(K-i)))  control=qubits[i], target=qubits[i+1]
             Splits amplitude: 1/(K-i) stays at i, (K-i-1)/(K-i) moves to i+1
          b. CX  control=qubits[i+1], target=qubits[i]
             Disentangles the control qubit, moving it back to |0>

    Verification for K=3:
        Start:                              |100>
        After CRy(2arccos(1/sqrt(3))) + CX: 1/sqrt(3)|100> + sqrt(2/3)|010>
        After CRy(pi/2) + CX:              1/sqrt(3)|100> + 1/sqrt(3)|010> + 1/sqrt(3)|001>

    Args:
        num_qubits (int): Total qubits in the system (N*K).
        N (int): Number of nodes.
        K (int): Number of communities.

    Returns:
        QuantumCircuit: The initial-state preparation circuit (no measurements).
    """
    qc = QuantumCircuit(num_qubits)

    for node in range(N):
        # Qubit indices for this node's K community slots
        qubits = [node * K + r for r in range(K)]

        # Place the excitation on the first qubit of this node's block
        qc.x(qubits[0])

        # Spread it uniformly across all K positions
        for i in range(K - 1):
            # After i steps, (K-i) positions remain to share the amplitude.
            # Rotation angle places 1/(K-i) of amplitude at qubits[i].
            angle = 2.0 * np.arccos(1.0 / np.sqrt(K - i))
            qc.cry(angle, qubits[i], qubits[i + 1])  # controlled-Ry
            qc.cx(qubits[i + 1], qubits[i])           # un-entangle control qubit

    return qc

In [ ]:
def create_qaoa_ansatz(total_hamiltonian, P, N, K, mixer_hamiltonian=None, initial_state=None):
    """Creates a QAOA ansatz circuit with a W-state initial state.

    Args:
        total_hamiltonian (SparsePauliOp): The problem Hamiltonian (cost operator).
        mixer_hamiltonian (SparsePauliOp): The XY mixer Hamiltonian.
        P (int): The number of QAOA layers (reps).

    Returns:
        QuantumCircuit: The constructed QAOA ansatz circuit with measurements.
    """
    num_qubits = total_hamiltonian.num_qubits

    # Build the W-state initial state: one W_K state per node's K-qubit block.
    # This ensures the circuit starts inside the one-hot subspace and the XY
    # mixer keeps it there throughout all P layers.
    if initial_state=="W": #Only for One-Hot
        initial_state = build_w_state_initial_state(num_qubits, N, K)

    # Initialize QAOAAnsatz with the W-state initial state and XY mixer.
    # Together these ensure the search stays within the valid one-hot subspace.
    circuit = QAOAAnsatz(cost_operator=total_hamiltonian, reps=P,
                         mixer_operator=mixer_hamiltonian,
                         initial_state=initial_state)
    circuit.measure_all()

    # Decompose into elementary gates that AerSimulator understands.
    circuit = circuit.decompose()
    return circuit

In [ ]:
def draw_circuit_mpl(circuit, title="Circuit"):
    """Draws the quantum circuit using Matplotlib.

    This function provides a visual representation of the quantum circuit,
    making it easier to understand its structure and operations.

    Args:
        circuit (qiskit.circuit.QuantumCircuit): The quantum circuit to draw.
        title (str, optional): The title to display above the circuit. Defaults to "Circuit".
    """
    # Draw the circuit using the 'mpl' (Matplotlib) backend.
    # fold=False prevents the circuit from being wrapped to multiple lines.
    fig = circuit.draw('mpl', fold=False)
    fig.suptitle(title) # Set the title of the Matplotlib figure
    display(fig) # Display the figure in the notebook

In [ ]:
def simulator_backend(method):
    if method == "statevector":
        return AerSimulator(seed_simulator=123)
    elif method == "mps":
        return AerSimulator(seed_simulator=123,
                           method="matrix_product_state",
                           matrix_product_state_max_bond_dimension=256,
                           matrix_product_state_truncation_threshold=1e-16)
    else:
        return AerSimulator(seed_simulator=123,
                           method="automatic")
    


def run_ansatz_simulator_circuit_prep(circuit, method="automatic"):
    """Prepares the quantum circuit for execution on a local AerSimulator.

    Uses generate_preset_pass_manager (post-Qiskit 0.44 Runtime-native API)
    instead of the legacy transpile() function. This ensures the returned
    circuit carries a proper .layout attribute, which EstimatorV2 requires
    via hamiltonian.apply_layout(ansatz.layout).

    Args:
        circuit (qiskit.circuit.QuantumCircuit): The quantum circuit to be prepared.

    Returns:
        tuple:
            - transpiled_qc (qiskit.circuit.QuantumCircuit): The transpiled circuit.
            - backend (qiskit_aer.AerSimulator): The AerSimulator instance.
    """
    # Strip measurements before transpilation — HighLevelSynthesis cannot synthesise
    # measure instructions. EstimatorV2 handles expectation values without them.
    circuit_no_meas = circuit.remove_final_measurements(inplace=False)

    backend = simulator_backend(method) # Initialize the AerSimulator for local simulation with a fixed seed for reproducibility.
    
    # Transpile the circuit to optimize it for the target simulator backend.
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,   # Level 1 is sufficient for simulation — fast
        backend=backend,
        seed_transpiler=123,
    )
    transpiled_qc = pass_manager.run(circuit_no_meas)
    return transpiled_qc, backend

In [ ]:
def cost_fun_estimator(params, ansatz, hamiltonian, estimator, eval_tracker):
    """Calculates the expectation value of the Hamiltonian for a given ansatz and parameters.

    This function serves as the objective function for the optimizer, returning the cost
    (expectation value) for a given set of QAOA parameters.

    Args:
        params (np.ndarray): The current parameters (gamma, beta) for the QAOA ansatz.
        ansatz (qiskit.circuit.QuantumCircuit): The QAOA ansatz circuit.
        hamiltonian (qiskit.quantum_info.SparsePauliOp): The total Hamiltonian (objective function).
        estimator (qiskit_ibm_runtime.EstimatorV2): The Qiskit Runtime Estimator primitive.
        eval_tracker (list): A list to track the cost values during optimization.

    Returns:
        float: The expectation value (cost) of the Hamiltonian for the given parameters.
    """
    # Apply the layout of the transpiled circuit to the Hamiltonian,
    # and extend the Hamiltonian to match the number of qubits in the ansatz.
    # This ensures compatibility between the ansatz and the observable for expectation value calculation.
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    # Create a Pub (Primitive Unified Bloc) for the Estimator, which bundles the ansatz,
    # observable (Hamiltonian), and parameters for a single estimation job.
    pub = (ansatz, isa_hamiltonian, params)

    # Run the Estimator job with the defined Pub.
    job = estimator.run([pub])

    # Retrieve the results from the Estimator job.
    results = job.result()[0]
    # Extract the expectation value (cost) from the results.
    cost = float(results.data.evs)

    # Track the cost value for plotting or analysis.
    eval_tracker.append(cost)

    return cost


def run_optimizer_simulator(transpiled_qc, backend, initial_params, total_hamiltonian):
    """Runs an optimization algorithm to find the best QAOA parameters.

    This function uses a classical optimizer (COBYLA) to minimize the cost function,
    which is the expectation value of the total Hamiltonian.

    Args:
        transpiled_qc (qiskit.circuit.QuantumCircuit): The transpiled QAOA ansatz circuit.
        backend (qiskit_aer.AerSimulator or qiskit_ibm_runtime.IBMQBackend): The quantum backend (simulator or real device).
        initial_params (list): Initial values for the QAOA parameters (gamma, beta).

    Returns:
        tuple: A tuple containing:
            - result (scipy.optimize.OptimizeResult): The result object from the optimization.
            - value_tracker (list): A list of cost values recorded during the optimization process.
    """
    value_tracker = [] # Initialize a list to store cost values at each optimization step.

    # Initialize the Estimator primitive with the specified backend.
    estimator = Estimator(mode = backend)
    # Set the default number of shots for the estimator and seed.
    estimator.options.default_shots = 4096
    estimator.options.seed_estimator = 123

    # Run the classical optimization using scipy.optimize.minimize.
    # The cost_fun_estimator is the objective function to minimize.
    # 'args' passes additional fixed arguments to the objective function.
    # 'method="COBYLA"' specifies the optimization algorithm.
    # 'tol' sets the tolerance for termination.
    result = minimize(
        cost_fun_estimator,
        initial_params,
        args = (transpiled_qc, total_hamiltonian, estimator, value_tracker),
        method = "COBYLA",
        #method='Powell',
        #method = 'L-BFGS-B',
        tol = 1e-2,
    )

    print(f"\n\nOptimizer Result: ")
    print(result)
    print(f"\n\nValue Tracker: ")
    print(value_tracker)

    return result, value_tracker

In [ ]:
def plot_optimizer_results(value_tracker):
    """Plots the optimization results, specifically the cost per iteration.

    This visualization helps in understanding the convergence behavior of the optimizer
    and how the cost function evolved over the optimization steps.

    Args:
        value_tracker (list): A list containing the cost values recorded at each
                              iteration of the optimization process.
    """
    # Create a new figure for the plot with a specified size.
    plt.figure(figsize=(12,6))
    # Plot the tracked cost values.
    plt.plot(value_tracker)
    # Label the x-axis as 'Iteration'.
    plt.xlabel("Iteration")
    # Label the y-axis as 'Cost'.
    plt.ylabel("Cost")
    # Display the plot.
    plt.show()

In [ ]:
def to_bitstring(integer, num_bits):
    """Converts an integer to a binary bitstring of a specified length."""
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


def sample_output(optimized_circuit_input, backend, num_qubits):
    """Samples measurement outcomes from an optimized quantum circuit and determines the most likely bitstring.

    Args:
        optimized_circuit_input (qiskit.circuit.QuantumCircuit): The quantum circuit with optimized parameters.
        backend (qiskit_aer.AerSimulator or qiskit_ibm_runtime.IBMQBackend): The quantum backend.
        num_qubits (int): Total number of qubits.

    Returns:
        list: The most likely bitstring as a list of integers (e.g., [0, 1, 0, 1]).
    """
    # The circuit may have had measurements stripped for EstimatorV2; re-add them.
    circuit = optimized_circuit_input.copy()
    if circuit.num_clbits == 0:
        circuit.measure_all()  # adds a classical register named 'meas'

    sampler = Sampler(mode=backend)
    sampler.options.default_shots = 10000

    job = sampler.run([circuit], shots=int(1e4))

    # Retrieve integer-based counts (for bitstring interpretation)
    counts_int = job.result()[0].data.meas.get_int_counts()
    # Retrieve binary string-based counts (for display)
    counts_bin = job.result()[0].data.meas.get_counts()
    shots = sum(counts_int.values())
    final_distribution_int = {key: val/shots for key, val in counts_int.items()}
    final_distribution_bin = {key: val/shots for key, val in counts_bin.items()}
    print(final_distribution_int)

    keys = list(final_distribution_int.keys())
    values = list(final_distribution_int.values())
    most_likely = keys[np.argmax(np.abs(values))]
    most_likely_bitstring = to_bitstring(most_likely, num_qubits)
    # Reverse to match Qiskit's qubit ordering (q0 is rightmost)
    most_likely_bitstring.reverse()

    print("Result bitstring:", most_likely_bitstring)

    return most_likely_bitstring

QAOA - QC Hardware implementation

$\text{Below code is when I want to run on IBM QC. However for this example, simulation will be done.}$

In [ ]:
def run_ansatz_qc_circuit_prep(circuit):
    """Prepares the quantum circuit for execution on an IBM Quantum backend.

    This function connects to the IBM Quantum service, selects a suitable backend,
    transpiles the input circuit for that backend, and returns the transpiled
    circuit along with the selected backend.

    - simulator=False in least_busy() ensures we land on a real QPU,
      not an IBM cloud simulator.
    - operational=True filters out backends that are down for maintenance.
    - Transpilation level stays at 3 (aggressive) — correct for real hardware
      where circuit depth directly increases noise.

    Args:
        circuit (qiskit.circuit.QuantumCircuit): The quantum circuit to be prepared.

    Returns:
        tuple: A tuple containing:
            - candidate_circuit (qiskit.circuit.QuantumCircuit): The transpiled circuit.
            - ibm_backend (qiskit_ibm_runtime.IBMQBackend): The selected IBM Quantum backend.
    """
    # Initialize the QiskitRuntimeService to connect to the IBM Quantum Platform.
    # This requires a channel, instance (CRN), and an API token.
    service = QiskitRuntimeService(
        channel='ibm_quantum_platform', #The channel specifies the IBM Quantum platform to connect to.
        instance='', #CRN goes here
        token=IBM_KEY #The API token is used for authentication to access IBM Quantum services.
    )
    # Select the least busy IBM Quantum backend with sufficient qubits for the circuit.
    ibm_backend = service.least_busy(
        min_num_qubits=circuit.num_qubits,
        simulator=False,
        operational=True
    )

    print(f"Selected backend: {ibm_backend.name}")
    
    # Generate a preset pass manager for transpilation, optimizing for the selected backend.
    # optimization_level=3 applies aggressive optimizations.
    pass_manager = generate_preset_pass_manager(
        optimization_level=3, 
        backend=ibm_backend
    )

    # Transpile the circuit using the generated pass manager.
    candidate_circuit = pass_manager.run(circuit)
    print(f"Transpiled depth: {candidate_circuit.depth()} (original: {circuit.depth()})")
    
    # Display the circuit (optional, can be commented out if not needed for debugging)
    draw_circuit_mpl(candidate_circuit, "Candidate Circuit")

    return candidate_circuit, ibm_backend

In [ ]:
def spsa_step(params, ansatz, hamiltonian, estimator, tracker,
              learning_rate=0.1, perturbation=0.1):
    """One SPSA gradient-estimation step.

    SPSA approximates the gradient as:
        g ≈ (f(θ+cΔ) - f(θ-cΔ)) / (2c) * Δ^{-1}
    where Δ is a random ±1 vector and c is the perturbation size.
    This requires exactly 2 circuit evaluations regardless of the
    number of parameters.

    Used as the objective for a manual SPSA loop below.
    """
    delta = np.random.choice([-1, 1], size=len(params))
    p_plus  = params + perturbation * delta
    p_minus = params - perturbation * delta
    f_plus  = cost_fun_estimator(p_plus,  ansatz, hamiltonian, estimator, [])
    f_minus = cost_fun_estimator(p_minus, ansatz, hamiltonian, estimator, [])
    gradient = (f_plus - f_minus) / (2 * perturbation) * (1.0 / delta)
    new_params = params - learning_rate * gradient
    cost = cost_fun_estimator(new_params, ansatz, hamiltonian, estimator, tracker)
    return new_params, cost


def transpile_for_hardware(circuit, ibm_backend):
    """Transpile a circuit for a real IBM Quantum backend.

    optimization_level=3 is the most thorough:
    - Searches for the best qubit layout (minimises SWAP insertions).
    - Applies gate cancellation and commutation rules.
    - Reduces circuit depth as much as possible.
    Deeper circuits accumulate more noise, so aggressive optimisation
    meaningfully improves solution quality on real hardware.

    Returns the ISA (Instruction Set Architecture) circuit.
    """
    pass_manager = generate_preset_pass_manager(
        optimization_level=3, backend=ibm_backend
    )
    return pass_manager.run(circuit)


def run_qaoa_hardware(circuit, cost_hamiltonian, initial_params,
                      n_iterations=50):
    """Run QAOA on the least-busy IBM Quantum backend.

    Steps
    -----
    1. Connect to IBM Quantum via QiskitRuntimeService.
    2. Pick the least-busy backend with enough qubits.
    3. Transpile the circuit to that backend's native gate set.
    4. Run a manual SPSA loop to optimise the parameters.
    5. Return the optimised parameters and cost tracker.

    Parameters
    ----------
    circuit : QuantumCircuit (logical, with measurements)
    cost_hamiltonian : SparsePauliOp
    initial_params : list[float]
    n_iterations : int
        Number of SPSA steps.  Keep small (20–50) to limit hardware time.

    Returns
    -------
    best_params : np.ndarray
    tracker : list[float]
    transpiled_qc : QuantumCircuit (ISA)
    ibm_backend : IBM backend object
    """
    # ── 1. Connect ────────────────────────────────────────────────────────────
    service = QiskitRuntimeService(
        channel='ibm_quantum_platform',   # use 'ibm_quantum_platform' for IBM Cloud CRN
        instance='crn:v1:bluemix:public:quantum-computing:us-east:a/20cbc447039d45a793162289ff4316c8:bad66d39-9c59-494d-80a6-3fffc760611c::',
        token=IBM_KEY,
    )

    # ── 2. Select backend ─────────────────────────────────────────────────────
    # least_busy picks the real device with the shortest job queue that has
    # at least as many qubits as our circuit.
    ibm_backend = service.least_busy(
        min_num_qubits=circuit.num_qubits,
        operational=True,
        simulator=False,   # exclude cloud simulators — we want real hardware
    )
    print(f"Selected backend: {ibm_backend.name}")

    # ── 3. Transpile ──────────────────────────────────────────────────────────
    transpiled_qc = transpile_for_hardware(circuit, ibm_backend)
    print(f"Transpiled depth: {transpiled_qc.depth()}  "
          f"(original: {circuit.depth()})")

    # ── 4. Optimise with SPSA ─────────────────────────────────────────────────
    # resilience_level=1 enables readout error mitigation — a calibration
    # circuit is run to characterise measurement errors and correct them.
    estimator = Estimator(mode=ibm_backend)
    estimator.options.default_shots = 4096
    estimator.options.resilience_level = 1   # readout error mitigation

    params = np.array(initial_params, dtype=float)
    tracker = []
    for i in range(n_iterations):
        # Decay learning rate and perturbation over iterations (standard SPSA schedule)
        lr = 0.1 / (1 + i * 0.05)
        c  = 0.1 / (1 + i * 0.01) ** 0.167
        params, cost = spsa_step(params, transpiled_qc, cost_hamiltonian,
                                 estimator, tracker,
                                 learning_rate=lr, perturbation=c)
        if (i + 1) % 10 == 0:
            print(f"  iter {i+1:3d}  ⟨H_C⟩ = {cost:.4f}")

    return params, tracker, transpiled_qc, ibm_backend


In [ ]:
def group_nodes_by_community(G, final_answer, no_of_node_in_supernode, N):
    """Groups nodes into communities based on the one-hot encoded bitstring output from QAOA.

    This function interprets the optimized bitstring, assigns each original graph node
    to a community, and then visualizes the graph with nodes colored by their assigned community.

    Args:
        final_answer (list): The most likely bitstring obtained from the QAOA circuit, as a
                            list of strings (e.g., ['0', '1', '0', '1']). Each '1' indicates
                            a node's assignment to a specific community slot.
        k (int): The number of communities.
    """

    # Define a list of colors for visualizing different communities.
    color_list = [
        '#e6194b', # Community 0 color
        '#3cb44b', # Community 1 color
        '#ffe119', # Community 2 color
        '#4363d8',
        '#f58231',
        '#911eb4',
        '#46f0f0',
        '#f032e6',
        '#bcf60c',
        '#fabebe',
        '#008080',
        '#e6beff',
        '#9a6324',
        '#fffac8',
        '#800000',
        '#aaffc3',
        '#808000',
        '#ffd8b1',
        '#000075',
        '#808080',
        '#ffffff',
        '#000000'
    ]

    # Ensure final_answer is a list of integers for numerical processing.
    final_bitstring_ints = [int(bit) for bit in final_answer]

    # Initialize an array to store community assignments for each node. -1 means unassigned.
    node_community_assignments = [-1] * N
    # Initialize a dictionary to map community indices to lists of nodes.
    community_to_nodes = {comm_idx: [] for comm_idx in range(no_of_node_in_supernode)}

    # Iterate through each original node to determine its community assignment.
    for node_idx in range(N):
        # Extract the k-bit segment corresponding to the current node from the bitstring.
        # This segment represents the one-hot encoding for the node's community.
        start_idx = node_idx * no_of_node_in_supernode
        end_idx = start_idx + no_of_node_in_supernode
        node_k_bits = final_bitstring_ints[start_idx:end_idx]

        assigned_community = -1
        # Find the community based on the one-hot encoding (which bit is '1').
        for comm_idx in range(no_of_node_in_supernode):
            if node_k_bits[comm_idx] == 1:
                assigned_community = comm_idx
                break

        # If a valid community assignment is found, record it.
        if assigned_community != -1:
            node_community_assignments[node_idx] = assigned_community
            community_to_nodes[assigned_community].append(node_idx)
        else:
            # Warn if a node is not assigned to exactly one community, indicating a suboptimal solution.
            print(f"Warning: Node {node_idx} has an invalid community assignment: {node_k_bits}")

    print("Node community assignments:", node_community_assignments)
    print("Community to nodes mapping:", community_to_nodes)

    # Prepare a list of colors for drawing the graph, initialized to a default 'gray'.
    node_colors = ['gray'] * N

    # Assign appropriate colors based on the determined community for each node.
    for node_idx, community_idx in enumerate(node_community_assignments):
        if community_idx != -1 and community_idx < len(color_list):
            node_colors[node_idx] = color_list[community_idx]
        elif community_idx >= len(color_list):
            print(f"Warning: Not enough colors for community {community_idx}. Using default color for node {node_idx}.")

    return node_colors